# Death Knight Lab · 03
## Train your first model

**Choose Run → Run All Cells.** No extra installation needed.

This time, the model really learns: it starts with random weights, plays games, and updates its move preferences from wins and losses. You will see a progress bar, a learning curve, and separate test results.

We are learning **how to play the 33-card subset**. We are not optimizing decks yet, and these results do not establish performance in full Standard.

In [ ]:
from pathlib import Path
import sys, importlib
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "lab.py").exists() and (p / "learner").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook in the death-knight-lab project folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import learner.policy, learner.training, learner.notebook
for module in (learner.policy, learner.training, learner.notebook):
    importlib.reload(module)
from learner.policy import FEATURE_NAMES
from learner.training import train
from learner.notebook import TrainingProgress, show_results, show_parameter_changes
print(f"Ready: a small policy with {len(FEATURE_NAMES)} learnable weights.")
print("New runs start with random weights. Existing runs resume their checkpoint.")

### Your settings

Start with **1,000 training games**. Evaluation games run in addition to training games. You can leave everything unchanged for your first run.

- **Same settings:** resume the same saved model.
- **Increase `TRAINING_GAMES`:** continue learning, for example from 1,000 to 2,000 games.
- **Change `RANDOM_SEED`:** start a new model with new decks.
- **Stop button:** interrupt; the latest fully completed game is saved. Rerun to continue.

Game counts and evaluation intervals must be even. Changing settings other than the training-game target creates a separate run.

In [ ]:
TRAINING_GAMES = 1000
EVALUATE_EVERY = 250
EVALUATION_GAMES = 100
RANDOM_SEED = 42
LEARNING_RATE = 0.01
RUN_CHECKS = True

### What the model is learning

This first model is a small **linear policy**, not a neural network. It scores legal actions using visible game facts: mana, health, board stats, cards, and possible targets. The feature descriptions are supplied by us; their preferences are learned. No expert moves or pretrained weights are used.

It receives **+1 for winning, −1 for losing, and 0 for a draw**. We also encourage exploration so it does not settle too quickly on one kind of move. The first 250 games use a random opponent; later games mix random play with frozen copies of a recent model checkpoint.

Training decks, monitoring decks, and final-test decks are separate. The model cannot see the opponent's hand or either shuffled deck order. Evaluation games never update the model.

In [ ]:
if RUN_CHECKS:
    import subprocess
    checked = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"],
        cwd=root, capture_output=True, text=True,
    )
    print(checked.stdout + checked.stderr)
    if checked.returncode:
        raise RuntimeError("Checks failed; training was not started.")
else:
    print("Checks skipped by your setting.")

### Start training

The bar counts completed **training games**. It briefly pauses for evaluations while the model is tested without updates. The blue line shows measured win rate against random play at each checkpoint—not training-game win rate. It may rise or fall.

Completed games save automatically. If an error occurs, the notebook stops rather than inventing a result.

In [ ]:
progress = TrainingProgress()
run = train(
    total_games=TRAINING_GAMES,
    eval_every=EVALUATE_EVERY,
    eval_games=EVALUATION_GAMES,
    seed=RANDOM_SEED,
    learning_rate=LEARNING_RATE,
    output_dir=root / "runs",
    progress=progress,
)

### Did it improve?

The first table uses **separate test decks and game seeds**, comparing the trained model with random play and the original untrained model. Each pairing is played twice with starting player reversed.

The uncertainty bounds can be wide, especially for small samples. A single good run is encouraging, but not proof of a strong Hearthstone player. The monitoring table shows how performance changed during this run.

In [ ]:
show_results(run)

### See what changed

These are the model parameters that moved most from their initial random values. Positive changes increase an action's score when that feature is present; negative changes lower it, holding other features fixed.

A card's parameter is **not** its standalone strength. Features interact, and the model only saw our small pool and opponent mix.

In [ ]:
show_parameter_changes(run)
changed = sum(abs(a-b) > 1e-10 for a,b in zip(run["initial"].weights, run["policy"].weights))
print(f"\n{changed} of {len(FEATURE_NAMES)} weights changed.")
print(f"Completed training games: {run['state']['completed']:,}")
print(f"Optimizer updates: {run['state']['optimizer']['steps']:,}")

### Your saved model

Both the model weights and the information needed to continue training are saved. Nothing is uploaded to GitHub automatically.

To keep going, change `TRAINING_GAMES` above to **2000**, then choose **Run → Run All Cells** again. That continues this model instead of restarting it. The displayed target is a total, not an additional game count.

In [ ]:
folder = Path(run["directory"])
print("Resume checkpoint:", folder / "checkpoint.json")
print("Model weights:", folder / f"model_{run['state']['completed']:06d}.json")

### What comes after this?

First, examine whether the model consistently beats its baselines across fresh seeds. Then we can begin **deck evolution**: freeze playing skill, compare candidate decks, make card swaps, and repeat. That will be a separate experiment; this notebook keeps the deck populations fixed.

The simulator is still experimental and needs more card coverage and live-game validation before making claims about actual Standard decks. For the implementation details, see `docs/training.md` in the project folder.